# 08｜Choice长期交易日历与主数据增量刷新验收

本Notebook完成四件事：

1. 将Choice沪深交易日历按可配置长期区间落入SQLite；
2. 保存Choice全A板块的当期成员快照；
3. 与上一期快照比较，记录新增、退出板块和字段变更；
4. 同一快照连续运行两次，验证主数据、日历、快照和变更表不会重复增长。

重要口径：`removed`只表示退出当前板块快照，**不自动等同于退市**。退市仍以Choice明确返回的退市日期为准。

运行前请确认：

- 已覆盖`qianji-data-mini 0.6.0`补丁；
- 已完整运行`00_openBB环境构建.ipynb`并重启内核；
- 当前内核可以导入`EmQuantAPI`并已用LoginActivator/token激活；
- `.env`中不要打印或分享Choice账号、密码、token或userinfo内容。


In [1]:
from pathlib import Path
import os
import sys


def locate_project_root():
    configured = os.getenv("QIANJI_PROJECT_ROOT", "").strip()
    candidates = [Path(configured)] if configured else []
    current = Path.cwd().resolve()
    candidates.extend([current, current.parent, *current.parents])
    for candidate in candidates:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "qianji_data_mini").exists():
            return candidate.resolve()
    raise RuntimeError("未找到项目根目录。请设置环境变量 QIANJI_PROJECT_ROOT。")


PROJECT_ROOT = locate_project_root()
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env", override=True)

print("Python路径：", sys.executable)
print("项目根目录：", PROJECT_ROOT)
print("Notebook当前目录：", Path.cwd().resolve())


Python路径： d:\minicoda3\envs\dm311\python.exe
项目根目录： D:\OneDrive\桌面\数据基座代码\qianji_openbb_mini
Notebook当前目录： D:\OneDrive\桌面\数据基座代码\qianji_openbb_mini\notebooks


In [2]:
from importlib.metadata import PackageNotFoundError, version
from packaging.version import Version

try:
    installed_version = version("qianji-data-mini")
except PackageNotFoundError:
    installed_version = "0.0.0"

print("qianji-data-mini版本：", installed_version)
if Version(installed_version) < Version("0.6.0"):
    raise RuntimeError(
        "当前版本低于0.6.0。请覆盖本次补丁，运行00号Notebook，彻底重启内核后再运行08号。"
    )

try:
    from EmQuantAPI import c
    print("EmQuantAPI：导入成功")
except Exception as exc:
    raise RuntimeError(f"EmQuantAPI导入失败：{type(exc).__name__}: {exc}") from exc


qianji-data-mini版本： 0.6.0
EmQuantAPI：导入成功


## 参数

默认使用“本机昨天”作为快照日，避免当天尚未收盘或板块尚未更新。长期日历默认从2010年开始。若合同权限或调用耗时有限，可先把开始日期改为近五年试跑。


In [3]:
from datetime import date, datetime, timedelta, timezone

SNAPSHOT_DATE = os.getenv(
    "CHOICE_REFRESH_SNAPSHOT_DATE",
    (date.today() - timedelta(days=1)).isoformat(),
)
CALENDAR_START_DATE = os.getenv("CHOICE_LONG_CALENDAR_START_DATE", "2010-01-01")
CALENDAR_END_DATE = os.getenv("CHOICE_REFRESH_CALENDAR_END_DATE", SNAPSHOT_DATE)
MARKETS = [
    item.strip().upper()
    for item in os.getenv("CHOICE_REFRESH_MARKETS", "CNSESH,CNSESZ").split(",")
    if item.strip()
]
SECTOR_CODE = os.getenv("CHOICE_ALL_A_SECTOR_CODE", "001004").strip()
UNIVERSE = os.getenv("CHOICE_UNIVERSE_NAME", "all_a_001004").strip()
BATCH_SIZE = int(os.getenv("CHOICE_MASTER_BATCH_SIZE", "100"))
CREATE_BACKUP = os.getenv("CHOICE_REFRESH_CREATE_BACKUP", "1") == "1"
RUN_IDEMPOTENCY_CHECK = os.getenv("CHOICE_REFRESH_RUN_TWICE", "1") == "1"
STRICT_MODE = os.getenv("CHOICE_REFRESH_STRICT", "0") == "1"

DATABASE_PATH = Path(
    os.getenv("QIANJI_DB_PATH", str(PROJECT_ROOT / "data" / "qianji_market.db"))
).resolve()
OUTPUT_DIR = PROJECT_ROOT / "validation_output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if date.fromisoformat(CALENDAR_START_DATE) > date.fromisoformat(CALENDAR_END_DATE):
    raise ValueError("长期日历开始日期不能晚于结束日期。")
if date.fromisoformat(CALENDAR_END_DATE) > date.fromisoformat(SNAPSHOT_DATE):
    raise ValueError("日历结束日期不应晚于快照日期。")

safe_config = {
    "snapshot_date": SNAPSHOT_DATE,
    "calendar_start_date": CALENDAR_START_DATE,
    "calendar_end_date": CALENDAR_END_DATE,
    "markets": MARKETS,
    "sector_code": SECTOR_CODE,
    "universe": UNIVERSE,
    "batch_size": BATCH_SIZE,
    "database_path": str(DATABASE_PATH),
    "choice_login_mode": os.getenv("CHOICE_LOGIN_MODE", "auto"),
    "choice_username_configured": bool(os.getenv("CHOICE_USERNAME")),
    "choice_password_configured": bool(os.getenv("CHOICE_PASSWORD")),
    "create_backup": CREATE_BACKUP,
    "run_idempotency_check": RUN_IDEMPOTENCY_CHECK,
}
safe_config


{'snapshot_date': '2026-09-01',
 'calendar_start_date': '2010-01-01',
 'calendar_end_date': '2026-09-01',
 'markets': ['CNSESH', 'CNSESZ'],
 'sector_code': '001004',
 'universe': 'all_a_001004',
 'batch_size': 100,
 'database_path': 'D:\\OneDrive\\桌面\\数据基座代码\\qianji_openbb_mini\\data\\qianji_market.db',
 'choice_login_mode': 'userInfo',
 'choice_username_configured': False,
 'choice_password_configured': False,
 'create_backup': True,
 'run_idempotency_check': True}

## 数据库备份与刷新前基线

备份使用SQLite自身的backup接口，可在数据库正在使用时生成一致性副本。备份文件不包含`.env`，但数据库本身属于公司数据，请按内部权限保存。


In [4]:
import sqlite3

from qianji_data_mini import Database

database = Database(DATABASE_PATH)
run_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup_path = None
if CREATE_BACKUP and DATABASE_PATH.exists():
    backup_dir = OUTPUT_DIR / "backups"
    backup_dir.mkdir(parents=True, exist_ok=True)
    backup_path = backup_dir / f"qianji_market_before_choice_refresh_{run_timestamp}.db"
    with database.connect() as source_connection, sqlite3.connect(backup_path) as target_connection:
        source_connection.backup(target_connection)
    print("数据库一致性备份：", backup_path)
else:
    print("数据库备份：未执行（数据库尚不存在或参数已关闭）")


def scalar(sql, params=()):
    with database.connect() as connection:
        return connection.execute(sql, params).fetchone()[0]


def scoped_counts(stage):
    market_placeholders = ",".join("?" for _ in MARKETS)
    return {
        "stage": stage,
        "security_master_choice": scalar(
            "SELECT COUNT(*) FROM security_master WHERE source='choice'"
        ),
        "calendar_scope": scalar(
            f"SELECT COUNT(*) FROM trading_calendar WHERE source='choice' "
            f"AND market IN ({market_placeholders}) AND trade_date BETWEEN ? AND ?",
            [*MARKETS, CALENDAR_START_DATE, CALENDAR_END_DATE],
        ),
        "current_snapshot": scalar(
            "SELECT COUNT(*) FROM security_universe_snapshot "
            "WHERE source='choice' AND universe=? AND snapshot_date=?",
            [UNIVERSE, SNAPSHOT_DATE],
        ),
        "current_changes": scalar(
            "SELECT COUNT(*) FROM security_master_change "
            "WHERE source='choice' AND universe=? AND snapshot_date=?",
            [UNIVERSE, SNAPSHOT_DATE],
        ),
        "refresh_run_log": scalar("SELECT COUNT(*) FROM reference_refresh_run"),
    }


counts_before = scoped_counts("刷新前")
counts_before


数据库一致性备份： D:\OneDrive\桌面\数据基座代码\qianji_openbb_mini\validation_output\backups\qianji_market_before_choice_refresh_20260902_090859.db


{'stage': '刷新前',
 'security_master_choice': 5213,
 'calendar_scope': 126,
 'current_snapshot': 0,
 'current_changes': 0,
 'refresh_run_log': 0}

## 第一次真实刷新

本单元会登录Choice并调用`sector`、`css`和两个市场的`tradedates`。不会输出账号、密码、token或userinfo。


In [5]:
from qianji_data_mini import refresh_choice_reference


refresh_kwargs = {
    "snapshot_date": SNAPSHOT_DATE,
    "calendar_start_date": CALENDAR_START_DATE,
    "calendar_end_date": CALENDAR_END_DATE,
    "markets": MARKETS,
    "sector_code": SECTOR_CODE,
    "universe": UNIVERSE,
    "batch_size": BATCH_SIZE,
    "database_path": DATABASE_PATH,
}

first_result = refresh_choice_reference(**refresh_kwargs)
counts_after_first = scoped_counts("第一次刷新后")

first_summary = first_result.model_dump(mode="json", exclude={"current_symbols"})
first_summary


[EmQuantAPI Python] [Em_Info][2026-09-02 09:09:00]:The current version is EmQuantAPI(V2.7.5.0).

[EmQuantAPI Python] [Em_Info][2026-09-02 09:09:00]:verifying your token...

[EmQuantAPI Python] [Em_Info][2026-09-02 09:09:00]:connect server...

[EmQuantAPI Python] [Em_Info][2026-09-02 09:09:02]:token login start success!

[EmQuantAPI Python] [Em_Info][2026-09-02 09:09:05]:updating ChoiceToHQ.xml from version 0 to 120

[EmQuantAPI Python] [Em_Info][2026-09-02 09:09:06]:loading ChoiceToHQ.xml...

[EmQuantAPI Python] [Em_Info][2026-09-02 09:09:08]:DownLoad D:/EMQuantAPI_Python/python3/libs/windows/bjse_code_conversion.txt success.

[EmQuantAPI Python] [Em_Info][2026-09-02 09:09:12]:percentflag(for csd/css/cses) update success.

[EmQuantAPI Python] [Em_Info][2026-09-02 09:10:47]:heartbeatthread end.



{'source': 'choice',
 'universe': 'all_a_001004',
 'snapshot_date': '2026-09-01',
 'previous_snapshot_date': '2026-08-31',
 'current_count': 5214,
 'added_symbols': ['301697.SZ', '601123.SH'],
 'removed_symbols': [],
 'modified_symbols': [],
 'unchanged_count': 5212,
 'security_received_rows': 5214,
 'security_stored_rows': 5214,
 'calendar_received_rows': 12176,
 'calendar_stored_rows': 12176,
 'errors': {},
 'started_at': '2026-09-02T01:08:59.157535Z',
 'finished_at': '2026-09-02T01:10:48.446873Z'}

## 第二次真实刷新：验证幂等

同一快照日重复运行后，主数据、日历、快照和变更明细行数应保持不变。`reference_refresh_run`运行日志增加一条是正常审计行为。


In [6]:
second_result = None
if RUN_IDEMPOTENCY_CHECK and not first_result.errors:
    second_result = refresh_choice_reference(**refresh_kwargs)
    print("第二次刷新完成。")
elif first_result.errors:
    print("第一次刷新有错误，为避免扩大调用，跳过第二次刷新。")
else:
    print("已通过参数关闭第二次刷新。")

counts_after_second = scoped_counts("第二次刷新后" if second_result else "未执行第二次刷新")
second_summary = (
    second_result.model_dump(mode="json", exclude={"current_symbols"})
    if second_result else None
)
second_summary


[EmQuantAPI Python] [Em_Info][2026-09-02 09:10:49]:The current version is EmQuantAPI(V2.7.5.0).

[EmQuantAPI Python] [Em_Info][2026-09-02 09:10:49]:verifying your token...

[EmQuantAPI Python] [Em_Info][2026-09-02 09:10:49]:connect server...

[EmQuantAPI Python] [Em_Info][2026-09-02 09:10:51]:token login start success!

[EmQuantAPI Python] [Em_Info][2026-09-02 09:10:53]:loading ChoiceToHQ.xml...

[EmQuantAPI Python] [Em_Info][2026-09-02 09:11:44]:heartbeatthread end.

第二次刷新完成。


{'source': 'choice',
 'universe': 'all_a_001004',
 'snapshot_date': '2026-09-01',
 'previous_snapshot_date': '2026-08-31',
 'current_count': 5214,
 'added_symbols': ['301697.SZ', '601123.SH'],
 'removed_symbols': [],
 'modified_symbols': [],
 'unchanged_count': 5212,
 'security_received_rows': 5214,
 'security_stored_rows': 5214,
 'calendar_received_rows': 12176,
 'calendar_stored_rows': 12176,
 'errors': {},
 'started_at': '2026-09-02T01:10:48.512175Z',
 'finished_at': '2026-09-02T01:11:44.484549Z'}

## 读取落库证据

以下查询全部读取SQLite，不再次调用Choice。


In [7]:
import json
import pandas as pd

database = Database(DATABASE_PATH)
current_snapshot = database.query_universe_snapshots(
    source="choice", universe=UNIVERSE, snapshot_date=SNAPSHOT_DATE
)
previous_snapshot_date = (
    first_result.previous_snapshot_date.isoformat()
    if first_result.previous_snapshot_date else None
)
previous_snapshot = (
    database.query_universe_snapshots(
        source="choice", universe=UNIVERSE, snapshot_date=previous_snapshot_date
    )
    if previous_snapshot_date else pd.DataFrame()
)
changes = database.query_master_changes(
    source="choice", universe=UNIVERSE, snapshot_date=SNAPSHOT_DATE
)
if not changes.empty:
    changes["changed_fields"] = changes["changed_fields"].map(
        lambda value: ",".join(json.loads(value or "[]"))
    )

all_master = database.query_security_master(source="choice")
current_symbols = set(current_snapshot["symbol"].tolist()) if not current_snapshot.empty else set()
current_master = all_master[all_master["symbol"].isin(current_symbols)].copy()

calendar = pd.concat(
    [
        database.query_trading_calendar(
            source="choice",
            market=market,
            start_date=CALENDAR_START_DATE,
            end_date=CALENDAR_END_DATE,
        )
        for market in MARKETS
    ],
    ignore_index=True,
)
refresh_runs = database.query_reference_refresh_runs().tail(10).copy()
idempotency = pd.DataFrame([counts_before, counts_after_first, counts_after_second])

print("当前成员快照：", len(current_snapshot))
print("上一期成员快照：", len(previous_snapshot), "日期：", previous_snapshot_date)
print("本期变更记录：", len(changes))
print("当前成员主数据：", len(current_master))
print("长期日历：", len(calendar))
display(idempotency)
display(changes.head(20))


当前成员快照： 5214
上一期成员快照： 5212 日期： 2026-08-31
本期变更记录： 2
当前成员主数据： 5214
长期日历： 12176


,stage,security_master_choice,calendar_scope,current_snapshot,current_changes,refresh_run_log
0,刷新前,5213,126,0,0,0
1,第一次刷新后,5215,12176,5214,2,1
2,第二次刷新后,5215,12176,5214,2,2


,source,universe,snapshot_date,symbol,change_type,previous_snapshot_date,changed_fields,detected_at
0,choice,all_a_001004,2026-09-01,301697.SZ,added,2026-08-31,membership,2026-09-02T01:11:44.341611+00:00
1,choice,all_a_001004,2026-09-01,601123.SH,added,2026-08-31,membership,2026-09-02T01:11:44.341611+00:00


## 日历与行情关联检查

长期日历按自然日保存：交易日`is_open=1`，休市日`is_open=0`。已有Choice日线如果落在本次日历范围内，应能匹配到对应市场的开市日。


In [8]:
calendar["parsed_date"] = pd.to_datetime(calendar["trade_date"], errors="coerce")
calendar["weekday"] = calendar["parsed_date"].dt.weekday

calendar_summary = (
    calendar.groupby("market", dropna=False)
    .agg(
        rows=("trade_date", "size"),
        unique_dates=("trade_date", "nunique"),
        first_date=("trade_date", "min"),
        last_date=("trade_date", "max"),
        open_days=("is_open", "sum"),
    )
    .reset_index()
)

with database.connect() as connection:
    daily_dates = pd.read_sql_query(
        "SELECT symbol, trade_date, adjustment FROM daily_bar "
        "WHERE source='choice' AND trade_date BETWEEN ? AND ? "
        "AND (symbol LIKE '%.SH' OR symbol LIKE '%.SZ') "
        "ORDER BY symbol, trade_date",
        connection,
        params=[CALENDAR_START_DATE, CALENDAR_END_DATE],
    )

if daily_dates.empty:
    daily_calendar_check = pd.DataFrame(
        columns=["symbol", "trade_date", "adjustment", "market", "is_open"]
    )
else:
    daily_dates["market"] = daily_dates["symbol"].map(
        lambda value: "CNSESH" if value.endswith(".SH") else "CNSESZ"
    )
    daily_calendar_check = daily_dates.merge(
        calendar[["market", "trade_date", "is_open"]],
        on=["market", "trade_date"],
        how="left",
    )

display(calendar_summary)
display(daily_calendar_check.head())


,market,rows,unique_dates,first_date,last_date,open_days
0,CNSESH,6088,6088,2010-01-01,2026-09-01,4047
1,CNSESZ,6088,6088,2010-01-01,2026-09-01,4047


,symbol,trade_date,adjustment,market,is_open
0,000001.SZ,2026-07-16,unadjusted,CNSESZ,1
1,000001.SZ,2026-07-17,unadjusted,CNSESZ,1
2,000001.SZ,2026-07-20,unadjusted,CNSESZ,1
3,000001.SZ,2026-07-21,unadjusted,CNSESZ,1
4,000001.SZ,2026-07-22,unadjusted,CNSESZ,1


## 质量门槛

“没有发生新增/退出/修改”是正常业务结果，不属于失败。质量门槛判断的是数据是否完整、关系是否自洽和重复刷新是否幂等。


In [9]:
expected_natural_days = (
    date.fromisoformat(CALENDAR_END_DATE) - date.fromisoformat(CALENDAR_START_DATE)
).days + 1
expected_calendar_rows = expected_natural_days * len(MARKETS)

snapshot_duplicates = int(
    current_snapshot.duplicated(["source", "universe", "snapshot_date", "symbol"]).sum()
) if not current_snapshot.empty else 0
calendar_duplicates = int(
    calendar.duplicated(["source", "market", "trade_date"]).sum()
) if not calendar.empty else 0
bad_calendar_dates = int(calendar["parsed_date"].isna().sum()) if not calendar.empty else 0
bad_open_flags = int((~calendar["is_open"].isin([0, 1])).sum()) if not calendar.empty else 0
weekend_open = int(((calendar["weekday"] >= 5) & (calendar["is_open"] == 1)).sum()) if not calendar.empty else 0
daily_not_open = int((daily_calendar_check["is_open"] != 1).sum()) if not daily_calendar_check.empty else 0

second_core_unchanged = (
    second_result is not None
    and all(
        counts_after_first[key] == counts_after_second[key]
        for key in [
            "security_master_choice", "calendar_scope", "current_snapshot", "current_changes"
        ]
    )
)
second_change_same = (
    second_result is not None
    and first_result.added_symbols == second_result.added_symbols
    and first_result.removed_symbols == second_result.removed_symbols
    and first_result.modified_symbols == second_result.modified_symbols
)

removed_status_preserved = True
removed_evidence = "本期无removed记录"
if first_result.removed_symbols and previous_snapshot_date:
    previous_master_values = {
        row["symbol"]: json.loads(row["master_json"] or "{}")
        for _, row in previous_snapshot.iterrows()
    }
    current_status = all_master.set_index("symbol")["status"].to_dict()
    comparisons = {
        symbol: (
            previous_master_values.get(symbol, {}).get("status"),
            current_status.get(symbol),
        )
        for symbol in first_result.removed_symbols
    }
    removed_status_preserved = all(before == after for before, after in comparisons.values())
    removed_evidence = f"核对{len(comparisons)}只removed证券，状态未被成员变化强制修改"

gates = []
def add_gate(name, passed, evidence):
    gates.append({"质量门槛": name, "通过": bool(passed), "证据": str(evidence)})

add_gate("第一次刷新无接口错误", not first_result.errors, first_result.errors or "errors={}")
add_gate("全A成员非空", first_result.current_count > 0, f"current_count={first_result.current_count}")
add_gate("主数据与成员数量一致", first_result.security_received_rows == first_result.current_count, f"master={first_result.security_received_rows}, members={first_result.current_count}")
add_gate("当前快照与成员数量一致", len(current_snapshot) == first_result.current_count, f"snapshot={len(current_snapshot)}")
add_gate("当前快照无重复主键", snapshot_duplicates == 0, f"duplicates={snapshot_duplicates}")
add_gate("当前成员均有主数据", len(current_master) == len(current_snapshot), f"master={len(current_master)}, snapshot={len(current_snapshot)}")
add_gate("成员分类数量关系成立", first_result.current_count == len(first_result.added_symbols) + len(first_result.modified_symbols) + first_result.unchanged_count, f"current={first_result.current_count}, added={len(first_result.added_symbols)}, modified={len(first_result.modified_symbols)}, unchanged={first_result.unchanged_count}")
add_gate("退出板块不被自动判定退市", removed_status_preserved, removed_evidence)
add_gate("长期日历自然日数量完整", len(calendar) == expected_calendar_rows, f"actual={len(calendar)}, expected={expected_calendar_rows}")
add_gate("各市场覆盖起止日期", len(calendar_summary) == len(MARKETS) and (calendar_summary["first_date"] == CALENDAR_START_DATE).all() and (calendar_summary["last_date"] == CALENDAR_END_DATE).all(), calendar_summary.to_dict("records"))
add_gate("长期日历无重复主键", calendar_duplicates == 0, f"duplicates={calendar_duplicates}")
add_gate("长期日历日期均可解析", bad_calendar_dates == 0, f"bad_dates={bad_calendar_dates}")
add_gate("开闭市标记仅为0或1", bad_open_flags == 0, f"bad_flags={bad_open_flags}")
add_gate("每个市场存在开市日", not calendar_summary.empty and (calendar_summary["open_days"] > 0).all(), calendar_summary[["market", "open_days"]].to_dict("records"))
add_gate("周末未错误标为开市", weekend_open == 0, f"weekend_open={weekend_open}")
add_gate("已有Choice日线均匹配开市日", daily_not_open == 0, f"checked={len(daily_calendar_check)}, not_open_or_missing={daily_not_open}")
add_gate("第二次刷新无接口错误", second_result is not None and not second_result.errors, (second_result.errors if second_result else "未执行"))
add_gate("第二次刷新核心表不增长", second_core_unchanged, idempotency.to_dict("records"))
add_gate("第二次刷新变更结论一致", second_change_same, "added/removed/modified两次一致" if second_change_same else "两次结果不一致或未执行")
add_gate("SQLite完整性检查通过", scalar("PRAGMA quick_check") == "ok", scalar("PRAGMA quick_check"))

quality_gates = pd.DataFrame(gates)
passed_count = int(quality_gates["通过"].sum())
failed_count = int((~quality_gates["通过"]).sum())
print(f"质量门槛：{passed_count}项通过，{failed_count}项失败")
display(quality_gates)


质量门槛：20项通过，0项失败


,质量门槛,通过,证据
0,第一次刷新无接口错误,True,errors={}
1,全A成员非空,True,current_count=5214
2,主数据与成员数量一致,True,"master=5214, members=5214"
3,当前快照与成员数量一致,True,snapshot=5214
4,当前快照无重复主键,True,duplicates=0
5,当前成员均有主数据,True,"master=5214, snapshot=5214"
6,成员分类数量关系成立,True,"current=5214, added=2, modified=0, unchanged=5212"
7,退出板块不被自动判定退市,True,本期无removed记录
8,长期日历自然日数量完整,True,"actual=12176, expected=12176"
9,各市场覆盖起止日期,True,"[{'market': 'CNSESH', 'rows': 6088, 'unique_da..."


## 数据地图更新与导出

本次导出包含可转交组长的硬证据；不会保存Choice密码、token或userinfo。


In [10]:
data_map = pd.DataFrame(
    [
        {"dataset": "security_master", "grain": "source+symbol", "purpose": "当前证券身份", "refresh": "每日/按需幂等覆盖", "source": "choice"},
        {"dataset": "trading_calendar", "grain": "source+market+trade_date", "purpose": "长期自然日开闭市", "refresh": "每日向后延展并允许历史修订", "source": "choice"},
        {"dataset": "security_universe_snapshot", "grain": "source+universe+snapshot_date+symbol", "purpose": "每期全A成员及主数据摘要", "refresh": "每日快照，可重复覆盖", "source": "choice"},
        {"dataset": "security_master_change", "grain": "source+universe+snapshot_date+symbol+change_type", "purpose": "新增/退出板块/字段变更", "refresh": "由相邻快照计算，可重复覆盖", "source": "choice"},
        {"dataset": "reference_refresh_run", "grain": "run_id", "purpose": "刷新审计证据", "refresh": "每次运行新增", "source": "choice"},
    ]
)

change_summary = pd.DataFrame(
    [
        {"change_type": "added", "count": len(first_result.added_symbols), "meaning": "本期进入板块"},
        {"change_type": "removed", "count": len(first_result.removed_symbols), "meaning": "本期退出板块，不等同退市"},
        {"change_type": "modified", "count": len(first_result.modified_symbols), "meaning": "成员仍在但主数据字段变化"},
        {"change_type": "unchanged", "count": first_result.unchanged_count, "meaning": "成员仍在且字段未变"},
    ]
)

overview = pd.DataFrame(
    [
        {"项目": "运行时间", "值": datetime.now(timezone.utc).isoformat()},
        {"项目": "数据库", "值": str(DATABASE_PATH)},
        {"项目": "备份", "值": str(backup_path) if backup_path else "未创建"},
        {"项目": "快照日", "值": SNAPSHOT_DATE},
        {"项目": "上一期快照日", "值": previous_snapshot_date or "首期基线"},
        {"项目": "当前成员数", "值": first_result.current_count},
        {"项目": "长期日历区间", "值": f"{CALENDAR_START_DATE} 至 {CALENDAR_END_DATE}"},
        {"项目": "长期日历行数", "值": len(calendar)},
        {"项目": "质量门槛", "值": f"{passed_count}通过/{failed_count}失败"},
    ]
)

snapshot_export_columns = [
    "source", "universe", "snapshot_date", "symbol", "captured_at"
]
current_snapshot_export = current_snapshot[snapshot_export_columns].copy() if not current_snapshot.empty else current_snapshot
previous_snapshot_export = previous_snapshot[snapshot_export_columns].copy() if not previous_snapshot.empty else previous_snapshot
calendar_export = calendar.drop(columns=["parsed_date", "weekday"], errors="ignore")

excel_path = OUTPUT_DIR / f"Choice长期交易日历主数据增量刷新验收_{run_timestamp}.xlsx"
json_path = OUTPUT_DIR / f"Choice长期交易日历主数据增量刷新验收_{run_timestamp}.json"

sheets = {
    "验收总览": overview,
    "质量门槛": quality_gates,
    "两次运行计数": idempotency,
    "变更汇总": change_summary,
    "变更明细": changes,
    "当前成员快照": current_snapshot_export,
    "上一期成员快照": previous_snapshot_export,
    "当前成员主数据": current_master,
    "长期交易日历": calendar_export,
    "日历汇总": calendar_summary,
    "行情日历关联": daily_calendar_check,
    "刷新运行日志": refresh_runs,
    "数据地图": data_map,
}

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    for sheet_name, frame in sheets.items():
        frame.to_excel(writer, sheet_name=sheet_name[:31], index=False)

from openpyxl import load_workbook
workbook = load_workbook(excel_path)
for worksheet in workbook.worksheets:
    worksheet.freeze_panes = "A2"
    worksheet.auto_filter.ref = worksheet.dimensions
    for column_cells in worksheet.columns:
        values = [str(cell.value or "") for cell in list(column_cells)[:200]]
        width = min(max(max((len(value) for value in values), default=8) + 2, 10), 50)
        worksheet.column_dimensions[column_cells[0].column_letter].width = width
workbook.save(excel_path)

json_payload = {
    "metadata": safe_config,
    "first_refresh": first_summary,
    "second_refresh": second_summary,
    "quality_summary": {"passed": passed_count, "failed": failed_count},
    "quality_gates": quality_gates.to_dict("records"),
    "idempotency_counts": idempotency.to_dict("records"),
    "change_summary": change_summary.to_dict("records"),
    "changes": changes.to_dict("records"),
    "calendar_summary": calendar_summary.to_dict("records"),
    "data_map": data_map.to_dict("records"),
}
json_path.write_text(json.dumps(json_payload, ensure_ascii=False, indent=2, default=str), encoding="utf-8")

print("Excel：", excel_path)
print("JSON：", json_path)


Excel： D:\OneDrive\桌面\数据基座代码\qianji_openbb_mini\validation_output\Choice长期交易日历主数据增量刷新验收_20260902_090859.xlsx
JSON： D:\OneDrive\桌面\数据基座代码\qianji_openbb_mini\validation_output\Choice长期交易日历主数据增量刷新验收_20260902_090859.json


## 最终结论

- 全部质量门槛通过：本次长期日历和主数据增量刷新可作为验收证据；
- `removed`大于0：需要人工核对板块调整、暂停上市、代码变化或退市事实，不能直接批量标记退市；
- 接口错误或数量不完整：先保存Excel/JSON中的错误码和错误信息，不要发送账号密码；
- 同一日期核心表增长：不要继续扩大范围，应先排查主键或覆盖逻辑。


In [11]:
print(f"最终结论：{passed_count}项通过，{failed_count}项失败")
if failed_count == 0:
    print("✅ 08号长期交易日历与主数据增量刷新验收通过。")
else:
    print("⚠️ 存在未通过项，请查看“质量门槛”工作表。")

if STRICT_MODE and failed_count:
    failed_names = quality_gates.loc[~quality_gates["通过"], "质量门槛"].tolist()
    raise RuntimeError(f"严格模式：以下质量门槛未通过：{failed_names}")


最终结论：20项通过，0项失败
✅ 08号长期交易日历与主数据增量刷新验收通过。
